In [0]:
%sql
SELECT `year` AS years, measure(`Total Orders`) AS Total_Order 
FROM edw_prod.prod.orders_semantic 
GROUP BY year

In [0]:
%sql
create or replace view edw_prod.prod.orders_semantic_test
with metrics 
language yaml 
as $$
version: 1.1

source: edw_prod.prod.fact_order

dimensions:
  - name: Year
    expr: YEAR(source.order_date)
    display_name: Year

measures:
  - name: Total Orders
    expr: COUNT(DISTINCT source.order_id)
    comment: Represents the total number of rows in the dataset. Use this measure
      to count all
    display_name: Total Orders
  - name: Total Products
    expr: COUNT(source.product_name)
    window:
      - order: Year
        semiadditive: last
        range: current
    display_name: Total Products

materialization:
  schedule: EVERY 12 HOURS
  mode: relaxed
  materialized_views:
    - name: yearly_sales
      type: aggregated
      dimensions:
        - Year
      measures:
        - Total Orders
$$